# Phase 1 — BFCL
Nhóm simple / multiple / parallel / irrelevance của BFCL (PLAN 3.2), chấm bằng AST checker chính thức của `bfcl-eval`. BFCL không có handler cho SmolLM2, nên `src/eval/bfcl.py` đăng ký một handler dùng đúng prompt tool của SFT. Model chạy qua vLLM: cần runtime GPU. Không cần token HF.


## 1. Cài thư viện
vLLM kéo theo torch và transformers của nó, nên notebook này không ghim transformers như các notebook khác. Cài xong phải khởi động lại session (Runtime → Restart session) vì torch đổi bản, rồi chạy tiếp từ mục 2. Colab báo xung đột với vài gói có sẵn (google-colab, cudf…); notebook này không dùng chúng.

In [ ]:
%pip install -q vllm==0.32.0 'requests' 'tqdm' 'numpy' 'pandas' 'huggingface_hub' 'pydantic>=2.8.2' 'python-dotenv>=1.0.1' 'tree_sitter==0.21.3' 'tree-sitter-java==0.21.0' 'tree-sitter-javascript==0.21.4' 'openai>=1.86.0' 'mistralai==1.7.0' 'anthropic>=0.75.0' 'cohere==5.18.0' 'typer>=0.12.5' 'tabulate>=0.9.0' 'datamodel-code-generator==0.25.7' 'google-genai>=1.52.0' 'qwen-agent' 'mpmath' 'tenacity>=8.5.0' 'writer-sdk>=2.1.0' 'overrides' 'boto3' 'beautifulsoup4' 'html2text' 'rank_bm25==0.2.2' 'google-search-results' 'sentence-transformers>=2.7.0' 'faiss-cpu' 'networkx' 'filelock'
%pip install -q --no-deps bfcl-eval==2026.3.23
from importlib.metadata import version
wanted = {"vllm": "0.32.0", "bfcl-eval": "2026.3.23"}
installed = {name: version(name) for name in wanted}
print(installed, "torch", version("torch"), "transformers", version("transformers"))
assert installed == wanted, "Phiên bản chưa khớp; chạy lại cell này."


## 2. Nạp mã nguồn của dự án

In [ ]:
import base64, hashlib, json, os, sys
from pathlib import Path

payload = base64.b64decode("...")
assert hashlib.sha256(payload).hexdigest() == "0ff9aeff8d2356bd62d4dc5b98e5f090a85cec6a89506d6f7d012bc7366880e9"
root = Path("/content/sft-base-playground")
root.mkdir(parents=True, exist_ok=True)
for relative, source in json.loads(payload).items():
    target = root / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding="utf-8")
os.chdir(root)
sys.path.insert(0, str(root))
# One GPU only (Kaggle gives two T4s): with two, Trainer splits each batch across them and the effective batch doubles.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
# Các object đã tạo (như lab) vẫn giữ nguyên; lần import tiếp dùng source mới.
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]
print("Source bundle SHA-256:", "0ff9aeff8d2356bd62d4dc5b98e5f090a85cec6a89506d6f7d012bc7366880e9")
print("Working directory:", root)


## 3. Kiểm tra mã
Chỉ chạy test của phần prompt, parser và BFCL; các test huấn luyện cần đúng bản transformers đã ghim.

In [ ]:
import subprocess
for pattern in ("test_bfcl.py", "test_scorers.py"):
    checks = subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", "tests", "-p", pattern, "-v"], capture_output=True, text=True)
    print(checks.stdout)
    print(checks.stderr)
    checks.check_returncode()


## 4. Chạy BFCL
`MODEL` là khóa trong `configs/models.yaml`, id trên Hub hoặc thư mục checkpoint. Kết quả ghi vào `results/bfcl/`.

In [ ]:
MODEL = "chat_reference"
!python -m src.eval.bfcl --model {MODEL}
